# Ebook RAG con LlamaIndex

Inspirado en [huggingface](https://huggingface.co/learn/cookbook/en/rag_llamaindex_librarian)

## Dependencias

Para correr los modelos de manera local.

In [ ]:
%pip install -q \
    llama-index \
    EbookLib \
    html2text \
    llama-index-embeddings-huggingface \
    llama-index-llms-ollama

In [ ]:
!apt install pciutils lshw

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
get_ipython().system_raw('ollama serve &')

Pull Llama2 from the Ollama library.

In [ ]:
!ollama pull llama2

## Documentos

Descargamos epubs de la pagina [Project Gutenberg](https://www.gutenberg.org/ebooks/subject/10163), bajo la tematica "Filosofia de la Religion" (que esten en ingles).

In [ ]:
!mkdir -p "./test/library"
!wget https://www.gutenberg.org/ebooks/41360.epub.noimages -O "./test/library/elementary_forms_religious_life-Durkheim.epub"
!wget https://www.gutenberg.org/ebooks/47025.epub.noimages -O "./test/library/essence_christianity-Feuerbach.epub"
!wget https://www.gutenberg.org/ebooks/30061.epub.noimages -O "./test/library/religious_sentiment-Brinton.epub"
!wget https://www.gutenberg.org/ebooks/47314.epub.noimages -O "./test/library/philosophy_natural_theology-Jackson.epub"
!wget https://www.gutenberg.org/ebooks/17802.epub.noimages -O "./test/library/myth_science-Vignoli.epub"
!wget https://www.gutenberg.org/ebooks/56354.epub.noimages -O "./test/library/authorize_bible_interpeters-Beecher.epub"
!wget https://www.gutenberg.org/ebooks/33677.epub.noimages -O "./test/library/sources_religious_insight-Royce.epub"
!wget https://www.gutenberg.org/ebooks/38446.epub.noimages -O "./test/library/philosophy_religion_psychology_history-Sabatier.epub"
!wget https://www.gutenberg.org/ebooks/30709.epub.noimages -O "./test/library/god_world-Robinson.epub"
!wget https://www.gutenberg.org/ebooks/55531.epub.noimages -O "./test/library/common_sense_religion-Beecher.epub"

## LlamaIndex

RAG with LlamaIndex, at its core, consists of the following broad phases:

1. **Loading**, de los documentos;
2. **Indexing**, vector embeddings;
3. **Querying**, para recuper los documentos relevantes.


### Loading

Naturally, let's start with the **loading** phase.

El objeto 
[`SimpleDirectoryReader`](https://docs.llamaindex.ai/en/stable/module_guides/loading/simpledirectoryreader.html)
soporta archivos en epub.

In [ ]:
from llama_index.core import SimpleDirectoryReader

loader = SimpleDirectoryReader(
    input_dir="./test/",
    recursive=True,
    required_exts=[".epub"],
)

documents = loader.load_data()

### Indexing

Se usa 
[`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-base-en-v1.5) to
para generar los embeddings.


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embedding_model,
)

### Querying

Para esto se usara Llama 2.


In [ ]:
from llama_index.llms.ollama import Ollama

llama = Ollama(
    model="llama2",
    request_timeout=40.0,
)

query_engine = index.as_query_engine(llm=llama)

## Prueba


In [ ]:
print(query_engine.query("What are the titles of all the books available? Show me the context used to derive your answer."))

In [ ]:
print(query_engine.query("What is religion?"))